# RAG

Um modelo responde com o que está no prompt. RAG é a técnica de buscar, a cada pergunta, os trechos de uma base que provavelmente contêm a resposta, e colocá-los no prompt antes de gerar. Este notebook percorre as etapas: representar texto como vetor, indexar um documento, recuperar os trechos, gerar a resposta com a fonte e, por fim, deixar o agente decidir quando buscar.

In [ ]:
# No Google Colab, descomente e rode uma vez.
# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph pdfplumber

import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image
from sklearn.decomposition import PCA

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_chroma import Chroma
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

In [ ]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0)
embeddings = init_embeddings("openai:text-embedding-3-small")

## Embeddings

Um embedding é um vetor que representa o significado de um texto. Textos próximos em significado ficam próximos no espaço, e é essa proximidade que a busca vai usar: em vez de procurar palavras iguais, ela procura vetores vizinhos.

In [ ]:
vector = embeddings.embed_query("A vacina reduziu os casos de gripe no inverno.")

print(len(vector))
print(vector[:5])

São 1536 números por texto, e o mesmo modelo de embedding sempre devolve vetores do mesmo tamanho. Os números em si não dizem nada; o que interessa é a distância entre dois vetores.

### Similaridade de cosseno

A medida usual é o cosseno do ângulo entre os dois vetores, que ignora o comprimento deles e olha só a direção:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}$$

Vale 1 para vetores na mesma direção e 0 para vetores sem relação. A função abaixo recebe dois textos, calcula os dois embeddings e aplica a fórmula.

In [ ]:
def cosine_similarity(text_a: str, text_b: str) -> float:
    """Similaridade de cosseno entre os embeddings de dois textos."""
    a, b = embeddings.embed_query(text_a), embeddings.embed_query(text_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print(round(cosine_similarity("A vacina reduziu os casos de gripe no inverno.", "A imunização diminuiu as infecções respiratórias na estação fria."), 2))

Duas frases sem nenhuma palavra em comum além do artigo ficam a 0,76, porque dizem a mesma coisa. A tabela abaixo compara a mesma frase de referência com frases de vários assuntos.

In [ ]:
REFERENCE = "A vacina reduziu os casos de gripe no inverno."

SENTENCES = [
    "A imunização diminuiu as infecções respiratórias na estação fria.",
    "Médicos recomendam a vacinação anual contra a gripe.",
    "A inflação ficou abaixo da expectativa do mercado.",
    "A seleção venceu o amistoso por dois a zero.",
    "O filme venceu o prêmio de melhor direção no festival.",
    "Telescópio registra a galáxia mais distante já observada.",
]

pd.DataFrame(
    [(text, round(cosine_similarity(REFERENCE, text), 2)) for text in SENTENCES],
    columns=["frase", "similaridade"],
).sort_values("similaridade", ascending=False)

A paráfrase fica em 0,76 e a frase sobre vacinação contra a gripe, que traz outro fato do mesmo assunto, em 0,64. As frases de economia, esporte, cinema e ciência ficam todas abaixo de 0,35, e a ordem entre elas pouco importa: o embedding separa o que fala do mesmo assunto do que não fala.

### Vizinhança por assunto

Com mais textos, a proximidade organiza o espaço em regiões. As doze manchetes abaixo vêm de três assuntos, e a projeção dos 1536 números em duas dimensões, feita por PCA, deixa ver se o embedding as separa. A `QUERY` entra no mesmo gráfico como uma estrela.

In [ ]:
HEADLINES = {
    "ciência": [
        "Novo satélite brasileiro é lançado para monitorar queimadas",
        "Pesquisadores medem derretimento recorde no Ártico",
        "Telescópio registra galáxia mais distante já observada",
        "Vacina experimental contra dengue passa na fase três",
    ],
    "economia": [
        "Bolsa fecha em alta após anúncio de corte de juros",
        "Inflação de agosto fica abaixo da expectativa do mercado",
        "Governo lança linha de crédito para energia solar",
        "Dólar recua com entrada de capital estrangeiro",
    ],
    "esporte": [
        "Seleção brasileira vence amistoso por dois a zero",
        "Time local contrata técnico português para a próxima temporada",
        "Maratona de Natal bate recorde de inscritos",
        "Tenista brasileira avança às quartas em Paris",
    ],
}

In [ ]:
QUERY = "Cientistas descobrem planeta parecido com a Terra"

texts = [headline for items in HEADLINES.values() for headline in items] + [QUERY]
labels = [category for category, items in HEADLINES.items() for _ in items]
points = PCA(n_components=2).fit_transform(embeddings.embed_documents(texts))

COLORS = {"ciência": "#2a78d6", "economia": "#eb6834", "esporte": "#1baf7a"}
fig, ax = plt.subplots(figsize=(6, 4))
for category in HEADLINES:
    rows = [i for i, label in enumerate(labels) if label == category]
    ax.scatter(points[rows, 0], points[rows, 1], c=COLORS[category], s=60, label=category)
ax.scatter(*points[-1], c="black", marker="*", s=220, label="QUERY")
ax.legend(frameon=False)
ax.set_axis_off()
plt.show()

Os três assuntos ocupam regiões diferentes, e a estrela caiu no meio das manchetes de ciência sem compartilhar nenhuma palavra com elas. Troque a `QUERY` por uma frase sobre juros ou sobre futebol e rode de novo: a estrela muda de região, e é essa distância que a busca vai calcular.

### Exercício 1

As dez sinopses abaixo estão em `MOVIES`. Escreva uma consulta sua, como um gênero ou uma situação, e ordene as sinopses pela `cosine_similarity` com ela. Qual filme ficou em primeiro, e a ordem faz sentido até que posição da lista?

In [ ]:
MOVIES = [
    "Um astronauta abandonado em Marte precisa cultivar comida para sobreviver até o resgate.",
    "Dois amigos de infância se reencontram em Lisboa e revivem um amor interrompido.",
    "Um detetive aposentado investiga o desaparecimento de uma pianista em Buenos Aires.",
    "Um robô de limpeza solitário encontra uma planta e muda o destino da humanidade.",
    "Soldados presos em uma praia esperam por barcos civis que cruzam o canal para salvá-los.",
    "Uma nadadora de cidade pequena treina para a seletiva olímpica contra a vontade da família.",
    "Uma família se muda para uma casa antiga e descobre que o porão guarda um segredo.",
    "Um professor de física tenta provar que viagens no tempo são possíveis usando um carro.",
    "Dois chefs rivais disputam a última estrela de um guia de restaurantes em Paris.",
    "Um garoto descobre que é um bruxo e vai estudar em uma escola escondida da vista de todos.",
]

# Seu código aqui

## Indexação

Indexar é preparar a base antes de qualquer pergunta, em três etapas: carregar o documento, quebrá-lo em trechos e guardar o embedding de cada trecho. O documento usado é o Regimento Geral da UFRN, um PDF público de mais de cem páginas.

### Loading

A unidade de texto no LangChain é o `Document`: um objeto com o texto em `page_content` e um dicionário livre em `metadata`. O metadado carrega o que não está no texto e vai ser preciso depois, como a origem, a página ou a data.

In [ ]:
manual = Document(
    page_content="O plenário do Departamento se reúne uma vez por mês.",
    metadata={"source": "anotação", "page": 1},
)

print(manual.page_content)
print(manual.metadata)

Um loader lê uma fonte e devolve uma lista de `Document` como esse, com os metadados que ele consegue extrair. Há um loader por tipo de fonte: `TextLoader` para arquivo de texto, `CSVLoader` para planilha, `WebBaseLoader` para página da web, `DirectoryLoader` para uma pasta inteira. Para PDF há mais de um, e o `PDFPlumberLoader` preserva bem o texto de páginas com colunas e tabelas.

In [ ]:
URL = "https://www.ufrn.br/resources/documentos/regimentos/RegimentoGeral.pdf"
FILE = Path("regimento.pdf")

if not FILE.exists():
    request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        FILE.write_bytes(response.read())

print(FILE.stat().st_size // 1024, "KB")

In [ ]:
docs = PDFPlumberLoader(str(FILE)).load()

print("documentos:", len(docs))
print("caracteres:", sum(len(doc.page_content) for doc in docs))
print("metadados:", {key: docs[30].metadata[key] for key in ("source", "page", "total_pages")})

O PDF virou uma lista de 139 `Document`, um por página, e o loader preencheu os metadados sozinho: o arquivo de origem, o número da página e o total. É esse metadado que permite dizer, no fim, de onde veio cada resposta.

### Chunking

Uma página inteira é um trecho grande demais para recuperar: mistura assuntos e gasta contexto à toa. O corte em trechos menores se chama chunking, e o tamanho do trecho é a decisão que mais afeta a qualidade da busca. Para ver onde cada cortador corta, as células abaixo aplicam quatro deles a uma única página, com trechos pequenos.

In [ ]:
sample = docs[9]

print(len(sample.page_content), "caracteres")
print(sample.page_content[:300])

In [ ]:
def show_chunks(pieces: list) -> None:
    """Imprime o tamanho e o começo de cada trecho."""
    for piece in pieces:
        print(f"{len(piece.page_content):4d} | {piece.page_content[:60]!r}")

O `RecursiveCharacterTextSplitter` tenta cortar em parágrafo; se o pedaço ainda passa do tamanho, tenta em linha, depois em frase, depois em palavra. A sobreposição faz cada trecho começar um pouco antes do fim do anterior.

In [ ]:
recursive = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

show_chunks(recursive.split_documents([sample]))

Nenhum trecho passou de 300 caracteres, e os cortes caíram em fim de linha, onde o texto já tinha uma quebra. O começo de cada trecho repete o fim do anterior por causa da sobreposição.

In [ ]:
by_line = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)

show_chunks(by_line.split_documents([sample]))

O `CharacterTextSplitter` corta em um único separador e junta as partes até encher o tamanho. Com a quebra de linha ele se parece com o recursivo nesta página; com um separador que o documento não tem, como a linha em branco, ele não consegue cortar e devolve a página inteira.

In [ ]:
by_token = TokenTextSplitter(chunk_size=80, chunk_overlap=10)

show_chunks(by_token.split_documents([sample]))

O `TokenTextSplitter` mede em tokens do modelo, que é a unidade que o prompt paga, e ignora a estrutura do texto: os cortes caem no meio de palavras e de frases. Oitenta tokens de português dão perto de 200 caracteres.

In [ ]:
MARKDOWN = """# Regulamento

Texto de abertura.

## Matrícula

Regras de matrícula.

## Avaliação

Regras de avaliação.

### Recuperação

Regras da recuperação."""

by_header = MarkdownHeaderTextSplitter([("#", "title"), ("##", "section"), ("###", "subsection")])

for piece in by_header.split_text(MARKDOWN):
    print(piece.metadata, "|", piece.page_content)

Quando o documento tem estrutura, o cortador pode usá-la. O `MarkdownHeaderTextSplitter` corta nos títulos e grava a hierarquia deles nos metadados de cada trecho, então o trecho da recuperação sabe que pertence à avaliação. O `HTMLHeaderTextSplitter` faz o mesmo com páginas web.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = text_splitter.split_documents(docs)

print("trechos:", len(chunks))
print("metadados preservados:", chunks[40].metadata["page"])

Para o índice, o recursivo com trechos de 1000 caracteres: as 139 páginas viraram 166 trechos, e cada um carrega os metadados da página de onde saiu. O recursivo é o padrão porque respeita o limite sem exigir que o documento tenha uma estrutura específica.

### Vector stores

Um vector store guarda cada trecho junto do seu embedding e responde à pergunta "quais vetores estão mais perto deste". O Chroma roda embutido no processo e grava em disco na pasta indicada. Há outros com a mesma interface: o `InMemoryVectorStore` do próprio LangChain, que vive só na memória e serve para experimentar, e o FAISS, uma biblioteca de busca vetorial rápida para índices grandes.

In [ ]:
vector_store = Chroma(
    collection_name="regimento",
    embedding_function=embeddings,
    persist_directory="./chroma_regimento",
    collection_metadata={"hnsw:space": "cosine"},
)

In [ ]:
ids = [f"chunk-{i}" for i in range(len(chunks))]
vector_store.add_documents(documents=chunks, ids=ids)

print("trechos indexados:", len(vector_store.get()["ids"]))

O `add_documents` calculou o embedding dos 166 trechos e gravou cada vetor junto do texto e dos metadados. Os `ids` fazem a operação ser idempotente: rodar a célula de novo substitui os mesmos 166 em vez de duplicá-los. O `hnsw:space` define a métrica da coleção como cosseno, a mesma da seção de embeddings; sem isso o Chroma usa distância euclidiana.

In [ ]:
QUESTION = "O que compete ao plenário do Departamento?"

reopened = Chroma(collection_name="regimento", embedding_function=embeddings, persist_directory="./chroma_regimento")

print("trechos no disco:", len(reopened.get()["ids"]))
print([doc.metadata["page"] for doc in reopened.similarity_search(QUESTION, k=3)])

Um objeto novo, criado só com o nome da coleção e a pasta, encontrou os 166 trechos gravados e respondeu à busca sem recalcular nenhum embedding. É o que separa um índice persistente de um em memória: o custo de indexar é pago uma vez, e a busca sobrevive ao reinício do processo.

### Exercício 2

Refaça o corte do regimento com `chunk_size` de 300 e de 2000, mantendo o mesmo `chunk_overlap`, e indexe cada versão em uma coleção própria do Chroma, com outro `collection_name`. Quantos trechos cada corte produziu, e em qual deles um artigo do regimento sobrevive inteiro dentro de um trecho?

In [ ]:
# Seu código aqui

## Retrieval

Recuperar é calcular o embedding da pergunta e devolver os `k` trechos mais próximos dela. O `k` é o orçamento: poucos trechos arriscam deixar a resposta de fora, muitos enchem o prompt de texto irrelevante e diluem o que importa.

In [ ]:
retrieved_docs = vector_store.similarity_search(QUESTION, k=3)

for doc in retrieved_docs:
    print(f'página {doc.metadata["page"]} | {doc.page_content[:70]!r}')

O `similarity_search` devolve uma lista de `Document`, do mais próximo ao mais distante, sem dizer o quanto cada um está perto. A versão com nota devolve pares de `Document` e número.

In [ ]:
results = vector_store.similarity_search_with_score(QUESTION, k=3)

for doc, score in results:
    print(f'{score:.3f} | página {doc.metadata["page"]} | {doc.page_content[:60]!r}')

A nota é a distância de cosseno entre a pergunta e o trecho, que vale 1 menos a similaridade da seção de embeddings: quanto menor, mais próximo. Ela serve para comparar trechos dentro de uma mesma busca, e não como medida absoluta: uma pergunta cuja resposta não está no documento também devolve três trechos, com as distâncias que houver.

### O que volta da busca

Cada item é o mesmo `Document` que saiu do loader e passou pelo cortador: o texto em `page_content`, o dicionário em `metadata` e o `id` que o índice recebeu.

In [ ]:
top = retrieved_docs[0]

print("campos:", sorted(vars(top)))
print("id:", top.id)
print("caracteres:", len(top.page_content))
print()
print(top.page_content)

O que o modelo vai receber é isso: o trecho inteiro, do tamanho que o cortador definiu na indexação, e não a página nem a frase. Se o trecho corta um artigo no meio, é o pedaço cortado que chega ao prompt, o que liga a qualidade da resposta diretamente à decisão de chunking.

### Metadados

O dicionário `metadata` acompanha o trecho desde o loader. Parte dele o loader calculou, e parte ele leu das propriedades do próprio arquivo.

In [ ]:
docs[30].metadata

`source`, `page` e `total_pages` são do loader; `Author`, `Title`, `Producer` e as datas vieram do PDF, e contam que o arquivo foi impresso de um `.docx` pelo Word em janeiro de 2020. O dicionário é um `dict` comum: dá para acrescentar chaves antes do `add_documents`, como a seção do documento ou a data de vigência, e tudo o que estiver nele volta junto do trecho na busca.

In [ ]:
early = vector_store.similarity_search(QUESTION, k=3, filter={"page": {"$lt": 20}})
late = vector_store.similarity_search(QUESTION, k=3, filter={"page": {"$gte": 100}})

print("páginas < 20  ->", [doc.metadata["page"] for doc in early])
print("páginas >= 100 ->", [doc.metadata["page"] for doc in late])

O filtro é um dicionário sobre os metadados, com operadores como `$lt`, `$gte` e `$eq`, e restringe os candidatos antes de ordenar por similaridade; por isso a mesma pergunta devolve trechos de regiões diferentes do documento. É assim que o metadado entra na busca: o embedding não sabe em que página, seção ou ano um trecho está, e o filtro é o que traz essa informação para a recuperação.

In [ ]:
for k in [1, 3, 6]:
    found = [doc.metadata["page"] for doc in vector_store.similarity_search(QUESTION, k=k)]
    print(f"k={k} -> páginas {found}")

Os primeiros trechos se mantêm quando `k` cresce, e os novos entram no fim da lista, mais distantes. A partir de algum ponto eles deixam de acrescentar informação e passam só a ocupar o prompt.

### Exercício 3

Escreva uma pergunta sua sobre o regimento e recupere os trechos com `k` igual a 2 e a 6. Depois repita a busca com um filtro de metadado que restrinja a um intervalo de páginas. Quantos dos trechos recuperados têm mesmo a ver com a sua pergunta, e o filtro ajudou ou atrapalhou?

In [ ]:
# Seu código aqui

## Geração

A última etapa monta o prompt com os trechos recuperados e pede a resposta. O template deixa a estrutura visível: a instrução vai na mensagem de sistema, e o contexto e a pergunta entram como variáveis da mensagem humana.

In [ ]:
SYSTEM_TEMPLATE = """Responda usando apenas o contexto e cite a página entre parênteses.
Seja breve: no máximo três frases.
Se o contexto não bastar, diga que não encontrou."""

HUMAN_TEMPLATE = """Contexto:
{context}

Pergunta: {question}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("human", HUMAN_TEMPLATE),
])


def format_docs(documents: list) -> str:
    """Junta os trechos em um texto só, com a página na frente de cada um."""
    return "\n\n".join(f"[página {doc.metadata['page']}] {doc.page_content}" for doc in documents)

In [ ]:
filled = prompt.invoke({"context": format_docs(retrieved_docs), "question": QUESTION})

for message in filled.messages:
    print(f"[{message.type}]")
    print(message.content[:400])
    print()

É isso que o modelo recebe: as regras em cima, os três trechos com as páginas no meio e a pergunta no fim. Quatro instruções da mensagem de sistema mudam o resultado: responder só com o contexto, citar a página, ser breve e dizer quando o contexto não basta.

In [ ]:
chain = prompt | model | StrOutputParser()


def answer(question: str, k: int = 3) -> str:
    """Recupera os trechos e gera a resposta citando a página."""
    documents = vector_store.similarity_search(question, k=k)
    return chain.invoke({"context": format_docs(documents), "question": question})

In [ ]:
print(answer(QUESTION))

A resposta veio do documento e traz a página, então dá para conferir. Esse é o ganho do RAG sobre perguntar direto ao modelo: a afirmação fica rastreável até a fonte.

In [ ]:
print(answer("Qual é a duração do mandato dos membros do Conselho Universitário?"))

Aqui a busca trouxe três trechos e nenhum responde, então o modelo diz que não encontrou em vez de inventar. A falha é da recuperação e não da geração, e sem a instrução de admitir a ausência o modelo preencheria a lacuna com algo plausível e errado.

### Exercício 4

Use o `answer` com duas perguntas suas: uma cuja resposta você sabe que está no regimento e outra que sabe que não está. Confira se a página citada na primeira contém mesmo a resposta, e o que o modelo fez na segunda.

In [ ]:
# Seu código aqui

## O retriever como ferramenta

Na função `answer` a busca acontece sempre, mesmo quando a pergunta não precisa dela. Embrulhando o retriever em uma ferramenta e entregando a um agente, a busca vira uma decisão do modelo: ele chama quando julga que precisa do documento e responde direto quando não precisa. O `create_agent` monta o laço de ferramentas; o que é deste notebook é a ferramenta.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


@tool
def search_regiment(query: str) -> str:
    """Busca trechos do Regimento Geral da UFRN relacionados à consulta."""
    return format_docs(retriever.invoke(query))

In [ ]:
SYSTEM = """Para perguntas sobre o regimento, use a ferramenta e cite a página entre parênteses.
Para o resto, responda direto.
Seja breve: no máximo três frases."""

rag_agent = create_agent(model, tools=[search_regiment], system_prompt=SYSTEM)
Image(rag_agent.get_graph().draw_mermaid_png())

In [ ]:
for question in [QUESTION, "Bom dia, tudo bem?"]:
    result = rag_agent.invoke({"messages": [HumanMessage(question)]})
    print("pergunta:", question)
    print("caminho :", " -> ".join(message.type for message in result["messages"]))
    print("resposta:", result["messages"][-1].content)
    print()

A pergunta sobre o regimento passou pela ferramenta, e a conversa registra a busca como uma `ToolMessage` entre o pedido e a resposta. A saudação foi respondida sem busca nenhuma. O grafo é o mesmo agente de duas caixas, `model` e `tools`; a diferença está no que a ferramenta faz quando é chamada, que agora é consultar o índice.

### Exercício 5

Rode o `rag_agent` com uma pergunta cuja resposta não está no regimento e observe a lista de mensagens: ele buscou? O que respondeu? Depois mude o `SYSTEM` para pedir que ele tente uma segunda busca com outras palavras antes de desistir, e diga o que mudou na lista.

In [ ]:
# Seu código aqui